# altVRAG Run 8 - hybrid-arm query cache (ColQwen on a Colab GPU)

**Self-contained: no repo upload needed.** This notebook encodes the 24
pre-selected Run 8 hybrid questions with **ColQwen2** (`vidore/colqwen2-v1.0`,
the same checkpoint as the index page vectors) into the exact
`query_cache/<sha1(text)>.json` format the laptop's `QueryVectorStore` reads,
then downloads `query_cache.zip`.

**Why a GPU**: the hybrid retrieval arm needs the *question side* encoded by
ColQwen (page-side vectors already exist in the local index). CPU encoding on
the laptop hangs; a free Colab T4 does it in seconds per question.

**Runtime**: ~5-10 min on a free T4 (mostly the one-time ~5 GB model download).

**After this notebook (on the laptop, repo root)**:
1. unzip the downloaded `query_cache.zip` into `data/` so the files land in
   `data/query_cache/*.json`
2. `python scripts/run_test8.py --check-cache` -> expect
   `hybrid cache coverage: 24/24`
3. `python scripts/run_test8.py --arms hybrid` (runs exactly the 24 hybrid rows)
4. `python scripts/make_test8_pdf.py results/demo/testrun8_<stamp> testRun8.pdf`


In [ ]:
!nvidia-smi || echo 'NO GPU VISIBLE -> Runtime > Change runtime type > T4 GPU'
import torch
print('torch', torch.__version__,
      '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 1. Install deps

`colpali-engine` brings the ColQwen2 model + processor; `torchao` is required
by peft to load the checkpoint's LoRA adapter (same recipe as the ingestion
notebook). If the `ColQwen2` import in the next cell fails, run
`%pip install -q -U transformers` and re-run it.


In [ ]:
%pip install -q colpali-engine pillow
!pip install -q -U torchao
import importlib
for mod in ('colpali_engine', 'transformers', 'peft', 'torchao'):
    try:
        m = importlib.import_module(mod)
        print(mod, getattr(m, '__version__', '?'))
    except Exception as e:
        print(mod, 'MISSING:', e)


## 2. Questions to encode

The 24 Run-8 hybrid questions are embedded below (copied verbatim from
`results/run8_hybrid_items.json`; string-exact match with `data/gold.json` was
verified by sha1). Optionally set `UPLOAD_GOLD = True` to extend the batch
with every question from a `gold.json` you upload - extra entries are
harmless (the laptop only reads the sha1 keys it asks for).


In [ ]:
# The exact 24 Run-8 hybrid questions (results/run8_hybrid_items.json).
HYBRID_QUESTIONS = [
    'Which achieved better performance in 2011, Organic Growth or Real Internal Growth?',
    'Which achieved better performance in the year Nestlé acquired Prometheus, Organic Growth or Real Internal Growth?',
    'What is the name of the brand which Nestlé acquired in 2000?',
    'What is the name of the brand which Nestlé acquired in the year Findus was divested?',
    'What is the percentage of the 10-year annual average of the Real Internal Growth for the company which has the tagline "Good Food, Good Life"?',
    'How many kinds of BICC Key Skills are there?',
    'How many kinds of Business Intelligence Competence Center Key Skills are there?',
    'What is the biggest reason why users sometimes choose not to use enterprise BI applications?',
    'What is the biggest reason why users sometimes choose not to use the applications, data of which are not mostly trusted by 8% users?',
    'What is the title of the presentation that ALCATEL-LUCENT made for the OVERVIEW of 2012?',
    'What is the title of the presentation that the company which had won 7 Nobel Prizes made for the OVERVIEW of 2012?',
    'Which region of the world accounts for the highest percentage of 2010 revenues?',
    'Which region of the world accounts for the highest percentage of revenues in the year 12% GROWTH is achieved?',
    'Where does Alcatel-Lucent have the IP Transformation Center in Europe?',
    'Where does the company which was awarded 2400 Patents in 2010 have the IP Transformation Center in Europe?',
    'What is the total percentage that wait Minutes and Hours for an average BI request?',
    'How many Mobile Operators are listed as References / proof of our expertise?',
    'Which country spend the most in Mobile display ads in 2013, according to IAB Europe AdEx Benchmark (2013)?',
    'Which country spend the most in Mobile display ads in the year 79300 Romanian use Twitter, according to IAB Europe AdEx Benchmark (2013)?',
    'What is the total percentage of 4G users with the 1 - 500 MB plan and the 501 MB - 1 GB plan in the UK?',
    'Which country had the higher percentage of Tablet Pageviews in June 2014, Russia or Hungary?',
    'Which country had the higher percentage of Tablet Pageviews in June 2014, Russia or the country where 61 million euros was spent in online display ads in 2013?',
    'How much is the Trading Operating Profit in the year Nestlé achieved the third largest Organic Growth in 10 years?',
    'In which year did the company that divested Eismann in 2004 achieve higher Organic Growth, 2003 or 2004?',
]

# Optional: extend with all questions from an uploaded gold.json.
UPLOAD_GOLD = False
if UPLOAD_GOLD:
    from google.colab import files
    import json as _json
    up = files.upload()  # pick data/gold.json
    for _name in up:
        _items = _json.loads(up[_name].decode('utf-8'))
        if isinstance(_items, dict):
            _items = _items.get('items', [])
        HYBRID_QUESTIONS += [str(it['question']).strip()
                             for it in _items if it.get('question')]

_seen, _uniq = set(), []
for _q in HYBRID_QUESTIONS:
    if _q not in _seen:
        _seen.add(_q)
        _uniq.append(_q)
HYBRID_QUESTIONS = _uniq
print(len(HYBRID_QUESTIONS), 'questions to encode')


## 3. Encode with ColQwen2

Mirrors `src/vvrag/retrieve/query_embed.py::ColQwenQueryEmbedder` exactly
(text query path, L2-normalized rows, word-split tokens) and writes the
`QueryVectorStore` format: one JSON per question keyed by
`sha1(question.strip())`, vectors rounded to 6 decimals.


In [ ]:
import hashlib, json, time
from pathlib import Path
import numpy as np
import torch
from colpali_engine.models import ColQwen2, ColQwen2Processor

MODEL_ID = 'vidore/colqwen2-v1.0'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise SystemExit('No GPU: Runtime > Change runtime type > T4 GPU')

def query_key(text: str) -> str:
    return hashlib.sha1(text.strip().encode('utf-8')).hexdigest()

OUT = Path('query_cache')
OUT.mkdir(exist_ok=True)

t0 = time.time()
processor = ColQwen2Processor.from_pretrained(MODEL_ID)
model = (ColQwen2.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
         .to(DEVICE).eval())
print(f'model ready on {DEVICE} in {time.time() - t0:.0f}s | dim',
      getattr(model.config, 'hidden_size', '?'), flush=True)

ok = 0
for i, q in enumerate(HYBRID_QUESTIONS, 1):
    p = OUT / f'{query_key(q)}.json'
    if p.exists():
        print(f'[{i}/{len(HYBRID_QUESTIONS)}] already cached {q[:70]!r}')
        ok += 1
        continue
    with torch.no_grad():
        batch = processor.process_queries([q]).to(DEVICE)
        emb = model(**batch)
        vecs = (emb[0].detach().cpu().float().numpy()
                if isinstance(emb, torch.Tensor)
                else np.asarray(emb[0], dtype=np.float32))
    if vecs.ndim != 2 or vecs.shape[0] == 0:
        print(f'[{i}/{len(HYBRID_QUESTIONS)}] FAIL shape {vecs.shape} {q[:60]!r}')
        continue
    norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9
    vecs = (vecs / norms).astype(np.float32)
    tokens = [t for t in q.lower().split() if t] or ['q']
    p.write_text(json.dumps({
        'text': q,
        'tokens': tokens[:vecs.shape[0]],
        'model': MODEL_ID,
        'device': str(DEVICE),
        'shape': list(vecs.shape),
        'vectors': vecs.round(6).tolist(),
    }, ensure_ascii=False), encoding='utf-8')
    ok += 1
    print(f'[{i}/{len(HYBRID_QUESTIONS)}] ok {vecs.shape} {q[:70]!r}', flush=True)
print(f'DONE: {ok}/{len(HYBRID_QUESTIONS)} encoded')

dims = set()
for f in OUT.glob('*.json'):
    dims.add(json.loads(f.read_text(encoding='utf-8'))['shape'][1])
print('dims present (index expects 128):', sorted(dims))


In [ ]:
import zipfile
from pathlib import Path

files_out = sorted(Path('query_cache').glob('*.json'))
with zipfile.ZipFile('query_cache.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files_out:
        z.write(f, f'query_cache/{f.name}')
print('json files:', len(files_out),
      '| zip bytes:', Path('query_cache.zip').stat().st_size)

try:
    from google.colab import files as colab_files
    colab_files.download('query_cache.zip')
except ImportError:
    print('Not running in Colab: download query_cache.zip manually.')


## 4. Done - laptop steps

1. Unzip `query_cache.zip` into the repo's `data/` directory
   (you should end up with `data/query_cache/<sha1>.json`, no extra nesting).
2. `python scripts/run_test8.py --check-cache`
   -> expect `hybrid cache coverage: 24/24`
3. `python scripts/run_test8.py --arms hybrid`
   -> runs exactly the 24 hybrid rows (model A, n=1, calibrated gate), then
   writes `summary.json`; use `--resume results/demo/testrun8_<stamp>` for any
   quota/network errors.
4. `python scripts/make_test8_pdf.py results/demo/testrun8_<stamp> testRun8.pdf`

**Troubleshooting**
- Coverage < 24: the question strings must match `data/gold.json`
  character-for-character (sha1 of the stripped text is the cache key). Use
  `UPLOAD_GOLD = True` above and encode straight from the real `gold.json`.
- OOM: pick a smaller batch? (batches here are 1); T4 is plenty for this model.
- `ColQwen2` import error: `%pip install -q -U transformers`, then re-run.
